# Trailing stop-loss overlay: backtest

Given a set of trades with known entries and exits, would an ATR-based trailing stop have improved them? Entries are assumed at the end of the day, the initial stop goes on at entry, and trailing starts from the next trading day. 5-minute bars are resampled to 15 minutes before the stop logic runs.

Everything here is synthetic: a random-walk price path and random trades.

In [ ]:
import os
from pathlib import Path

# All inputs and outputs live under DATA_DIR (relative to the repo by default).
DATA_DIR = Path(os.environ.get("DATA_DIR", "data"))
OUT_DIR = DATA_DIR / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42

In [ ]:
import numpy as np
import pandas as pd


def synthetic_bars(n_days=250, seed=SEED, session=("09:30", "16:00")):
    """5-minute OHLC bars for one instrument, random walk with mild volatility clustering."""
    rng = np.random.default_rng(seed)
    frames, price, vol = [], 100.0, 0.0012
    for day in pd.bdate_range("2023-01-02", periods=n_days):
        idx = pd.date_range(f"{day.date()} {session[0]}", f"{day.date()} {session[1]}", freq="5min", inclusive="left")
        vol = 0.9 * vol + 0.1 * rng.uniform(0.0006, 0.002)
        close = price * np.exp(np.cumsum(rng.normal(0, vol, len(idx))))
        open_ = np.r_[price, close[:-1]]
        high = np.maximum(open_, close) * (1 + rng.uniform(0, vol, len(idx)))
        low = np.minimum(open_, close) * (1 - rng.uniform(0, vol, len(idx)))
        frames.append(pd.DataFrame(dict(open=open_, high=high, low=low, close=close), index=idx))
        price = close[-1]
    return pd.concat(frames)


def to_15m(m5):
    agg = {"open": "first", "high": "max", "low": "min", "close": "last"}
    return m5.resample("15min", label="left", closed="left").agg(agg).dropna()


def atr(bars, period=14):
    prev = bars.close.shift()
    tr = pd.concat([bars.high - bars.low, (bars.high - prev).abs(), (bars.low - prev).abs()], axis=1).max(axis=1)
    return tr.rolling(period).mean()


def synthetic_trades(bars, n=120, seed=SEED, max_hold_days=10):
    """End-of-day entries with a random direction and a fixed holding period (the 'original' exit)."""
    rng = np.random.default_rng(seed + 1)
    days = pd.Series(bars.index.normalize().unique())
    picks = np.sort(rng.choice(len(days) - max_hold_days - 1, n, replace=False))
    rows = []
    for i, p in enumerate(picks):
        d0, d1 = days[p], days[p + rng.integers(2, max_hold_days)]
        e = bars.loc[bars.index.normalize() == d0].iloc[-1]
        x = bars.loc[bars.index.normalize() == d1].iloc[-1]
        rows.append(dict(trade_id=i, direction=rng.choice([1, -1]), entry_time=e.name, entry_price=e.close,
                         exit_time=x.name, exit_price=x.close))
    return pd.DataFrame(rows)


def simulate_trailing_stop(trade, bars, atr_series, initial_pct, atr_mult, cap_pct):
    """Initial stop at initial_pct from entry; from the next day on, trail at atr_mult * ATR,
    never looser than cap_pct from the best price. Exit at the stop or at the original exit."""
    d = trade.direction
    window = bars.loc[(bars.index > trade.entry_time) & (bars.index <= trade.exit_time)]
    window = window[window.index.normalize() > trade.entry_time.normalize()]  # next day only
    stop = trade.entry_price * (1 - d * initial_pct)
    best = trade.entry_price
    for ts, bar in window.iterrows():
        hit = bar.low <= stop if d == 1 else bar.high >= stop
        if hit:
            fill = min(bar.open, stop) if d == 1 else max(bar.open, stop)  # gap-through fills at the open
            return ts, fill, "stop"
        best = max(best, bar.high) if d == 1 else min(best, bar.low)
        a = atr_series.get(ts, np.nan)
        if not np.isnan(a):
            trail = best - d * atr_mult * a
            cap = best * (1 - d * cap_pct)
            trail = max(trail, cap) if d == 1 else min(trail, cap)
            stop = max(stop, trail) if d == 1 else min(stop, trail)
    return trade.exit_time, trade.exit_price, "original"


def run_overlay(trades, bars, atr_series, **params):
    out = []
    for t in trades.itertuples(index=False):
        ts, px, why = simulate_trailing_stop(t, bars, atr_series, **params)
        out.append(dict(trade_id=t.trade_id, new_exit_time=ts, new_exit_price=px, exit_reason=why,
                        original_ret=t.direction * (t.exit_price / t.entry_price - 1),
                        overlay_ret=t.direction * (px / t.entry_price - 1)))
    return pd.DataFrame(out)


def summarise(res, col):
    r = res[col]
    return pd.Series({"trades": len(r), "hit_rate": (r > 0).mean(), "mean_ret": r.mean(),
                      "worst": r.min(), "sum_ret": r.sum()})

In [ ]:
m5 = synthetic_bars()
bars = to_15m(m5)
atr15 = atr(bars, period=14)
trades = synthetic_trades(bars)
trades.head()

In [ ]:
PARAMS = dict(initial_pct=0.01, atr_mult=3.0, cap_pct=0.03)
res = run_overlay(trades, bars, atr15, **PARAMS)
summary = pd.DataFrame({"original": summarise(res, "original_ret"), "with_trailing_stop": summarise(res, "overlay_ret")})
print(res.exit_reason.value_counts())
summary

In [ ]:
res.to_csv(OUT_DIR / "trailing_stop_trades.csv", index=False)
summary.to_csv(OUT_DIR / "trailing_stop_summary.csv")